In [17]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import scipy.io as sio

# Vérification du GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation de : {device}")

Utilisation de : cpu


In [9]:
class NTUFiDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.classes = ['box', 'circle', 'clean', 'fall', 'run', 'walk']
        self.data = []
        
        for idx, cls in enumerate(self.classes):
            cls_path = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_path): continue
            
            for file in os.listdir(cls_path):
                if file.endswith('.mat'):
                    self.data.append((os.path.join(cls_path, file), idx))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        file_path, label = self.data[idx]
        # Chargement de la variable 'CSIamp' identifiée
        mat = sio.loadmat(file_path)
        csi = mat['CSIamp'].astype(np.float32) # Shape (342, 2000)
        
        # Ajout de la dimension canal : (1, 342, 2000)
        csi = torch.from_numpy(csi).unsqueeze(0)
        return csi, label

In [10]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, ratio=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // ratio, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out) [cite: 327]

class TimeSubcarrierAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False) [cite: 336]
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(x_cat)) [cite: 334]

class CTSCAB(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.ca1 = ChannelAttention(in_channels)
        self.tsa = TimeSubcarrierAttention()
        self.ca2 = ChannelAttention(in_channels)

    def forward(self, x):
        x = x * self.ca1(x)
        x = x * self.tsa(x)
        x = x * self.ca2(x) [cite: 311]
        return x

class LI_HAR_Net(nn.Module):
    def __init__(self):
        super().__init__()
        layers = []
        in_c = 1
        # 5 blocs : Conv(3x3, 64) -> BN -> ReLU -> MaxPool [cite: 281-282]
        for _ in range(5):
            layers += [
                nn.Conv2d(in_c, 64, kernel_size=3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2)
            ]
            in_c = 64
        self.features = nn.Sequential(*layers)
        self.attention = CTSCAB(64) [cite: 72]

    def forward(self, x):
        x = self.features(x)
        x = self.attention(x)
        return x.view(x.size(0), -1)

In [11]:
def euclidean_dist(x, y):
    n, m, d = x.size(0), y.size(0), x.size(1)
    x = x.unsqueeze(1).expand(n, m, d)
    y = y.unsqueeze(0).expand(n, m, d)
    return torch.pow(x - y, 2).sum(2)

def train_episode(model, optimizer, data, labels, n_way, n_support, n_query):
    model.train()
    optimizer.zero_grad()
    
    # Sélection des classes pour l'épisode
    classes = torch.unique(labels)
    selected_cls = classes[torch.randperm(len(classes))[:n_way]]
    
    support_features, query_features = [], []
    query_labels = []

    for idx, c in enumerate(selected_cls):
        cls_data = data[labels == c]
        perm = torch.randperm(len(cls_data))
        
        # Support et Query
        sup = cls_data[perm[:n_support]]
        que = cls_data[perm[n_support:n_support+n_query]]
        
        support_features.append(model(sup).mean(0)) # Prototype [cite: 348]
        query_features.append(model(que))
        query_labels.append(torch.full((n_query,), idx, dtype=torch.long))

    prototypes = torch.stack(support_features)
    query_features = torch.cat(query_features)
    query_labels = torch.cat(query_labels).to(device)
    
    dists = euclidean_dist(query_features, prototypes)
    log_p_y = F.log_softmax(-dists, dim=1)
    
    loss = F.nll_loss(log_p_y, query_labels)
    loss.backward()
    optimizer.step()
    
    acc = (log_p_y.argmax(1) == query_labels).float().mean()
    return loss.item(), acc.item()

In [12]:
def euclidean_dist(x, y):
    n, m, d = x.size(0), y.size(0), x.size(1)
    x = x.unsqueeze(1).expand(n, m, d)
    y = y.unsqueeze(0).expand(n, m, d)
    return torch.pow(x - y, 2).sum(2)

def train_episode(model, optimizer, data, labels, n_way, n_support, n_query):
    model.train()
    optimizer.zero_grad()
    
    # Sélection des classes pour l'épisode
    classes = torch.unique(labels)
    selected_cls = classes[torch.randperm(len(classes))[:n_way]]
    
    support_features, query_features = [], []
    query_labels = []

    for idx, c in enumerate(selected_cls):
        cls_data = data[labels == c]
        perm = torch.randperm(len(cls_data))
        
        # Support et Query
        sup = cls_data[perm[:n_support]]
        que = cls_data[perm[n_support:n_support+n_query]]
        
        support_features.append(model(sup).mean(0)) # Prototype [cite: 348]
        query_features.append(model(que))
        query_labels.append(torch.full((n_query,), idx, dtype=torch.long))

    prototypes = torch.stack(support_features)
    query_features = torch.cat(query_features)
    query_labels = torch.cat(query_labels).to(device)
    
    dists = euclidean_dist(query_features, prototypes)
    log_p_y = F.log_softmax(-dists, dim=1)
    
    loss = F.nll_loss(log_p_y, query_labels)
    loss.backward()
    optimizer.step()
    
    acc = (log_p_y.argmax(1) == query_labels).float().mean()
    return loss.item(), acc.item()

In [15]:
import os

# 1. Vérification du dossier de base (CORRIGÉ : on enlève 'test1/')
base_path = 'NTU-Fi_HAR/train_amp'

print(f"Dossier de travail actuel : {os.getcwd()}")
print(f"Le chemin '{base_path}' existe-t-il ? : {os.path.exists(base_path)}")

if os.path.exists(base_path):
    print("\n✅ Le chemin a été trouvé ! Analyse du contenu :")
    # 2. Inspection des sous-dossiers
    for folder in os.listdir(base_path):
        folder_path = os.path.join(base_path, folder)
        
        # On vérifie que c'est bien un dossier (et pas un fichier caché par exemple)
        if os.path.isdir(folder_path):
            files = os.listdir(folder_path)
            print(f"\n📂 Dossier : {folder} | Contient {len(files)} fichiers.")
            
            if len(files) > 0:
                print(f"  -> Exemple de fichier : {files[0]}")
            else:
                print("  -> Attention : Ce dossier est vide.")
else:
    print("\n❌ ERREUR : Le chemin n'existe toujours pas. Vérifiez que le dossier 'NTU-Fi_HAR' est bien présent dans votre dossier actuel.")

Dossier de travail actuel : c:\Users\ilyas\Desktop\test1
Le chemin 'NTU-Fi_HAR/train_amp' existe-t-il ? : True

✅ Le chemin a été trouvé ! Analyse du contenu :

📂 Dossier : box | Contient 156 fichiers.
  -> Exemple de fichier : box0.mat

📂 Dossier : circle | Contient 156 fichiers.
  -> Exemple de fichier : circle0.mat

📂 Dossier : clean | Contient 156 fichiers.
  -> Exemple de fichier : clean0.mat

📂 Dossier : fall | Contient 156 fichiers.
  -> Exemple de fichier : fall0.mat

📂 Dossier : run | Contient 156 fichiers.
  -> Exemple de fichier : run0.mat

📂 Dossier : walk | Contient 156 fichiers.
  -> Exemple de fichier : walk0.mat


In [18]:
# ---------------------------------------------------------
# Cellule d'Entraînement Principale
# ---------------------------------------------------------

# 1. Chargement des données avec le chemin corrigé
dataset_path = 'NTU-Fi_HAR/train_amp'
train_dataset = NTUFiDataset(dataset_path)

print(f"📊 Nombre total d'échantillons trouvés : {len(train_dataset)}")

# 2. Sécurité : On ne lance l'entraînement que si des données ont été trouvées
if len(train_dataset) == 0:
    print("❌ L'entraînement est annulé car le dataset est vide.")
    print("Assurez-vous que le dossier 'NTU-Fi_HAR/train_amp' contient bien des dossiers de classes avec des fichiers .mat.")
else:
    # 3. Préparation du DataLoader
    # On charge tout le dataset d'un coup pour faciliter l'échantillonnage par épisode
    train_loader = DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=True)

    # 4. Initialisation du modèle et de l'optimiseur
    model = LI_HAR_Net().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print("\n🚀 Démarrage de l'entraînement (Few-Shot)...")
    epochs = 40
    
    # Mode entraînement
    model.train()
    
    for epoch in range(epochs):
        # On récupère toutes nos données d'un coup depuis le loader
        for data, labels in train_loader:
            data, labels = data.to(device), labels.to(device)
            
            # Entraînement sur un épisode (6 classes, 5-shot, 5 requêtes de test)
            # Vous pouvez ajuster n_support et n_query selon la taille de vos dossiers
            loss, acc = train_episode(
                model=model, 
                optimizer=optimizer, 
                data=data, 
                labels=labels, 
                n_way=6,       # Les 6 activités (box, circle, clean, fall, run, walk)
                n_support=5,   # Nombre d'échantillons pour calculer le prototype
                n_query=5      # Nombre d'échantillons pour évaluer la loss
            )
        
        # Affichage des résultats tous les 5 epochs pour suivre la progression
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"Époque [{epoch+1}/{epochs}] | Loss: {loss:.4f} | Précision (Accuracy): {acc:.4f}")
            
    print("\n✅ Entraînement terminé avec succès !")

📊 Nombre total d'échantillons trouvés : 936


NameError: name 'cite' is not defined

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import scipy.io as sio

# =====================================================================
# 1. Configuration initiale
# =====================================================================
# Utilisation du GPU si disponible, sinon CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🖥️ Appareil utilisé : {device}")

# =====================================================================
# 2. Classe Dataset pour charger vos données NTU-Fi_HAR
# =====================================================================
class NTUFiDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir
        # Vos 6 classes d'activités
        self.classes = ['box', 'circle', 'clean', 'fall', 'run', 'walk']
        self.data = []
        
        # Balayage des dossiers pour trouver les fichiers .mat
        for idx, cls in enumerate(self.classes):
            cls_path = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_path): 
                continue
            
            for file in os.listdir(cls_path):
                if file.endswith('.mat'):
                    self.data.append((os.path.join(cls_path, file), idx))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        file_path, label = self.data[idx]
        
        # Chargement du fichier .mat et extraction de 'CSIamp'
        mat = sio.loadmat(file_path)
        csi = mat['CSIamp'].astype(np.float32) # Dimension attendue : (342, 2000)
        
        # Ajout de la dimension "Canal" (Channel) requise par PyTorch -> (1, 342, 2000)
        csi = torch.from_numpy(csi).unsqueeze(0)
        
        return csi, label

# =====================================================================
# 3. Architecture du Modèle (CNN + CTSC-AB)
# =====================================================================
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, ratio=8):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // ratio, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(in_channels // ratio, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        return self.sigmoid(avg_out + max_out) # [cite: 327]

class TimeSubcarrierAttention(nn.Module):
    def __init__(self):
        super().__init__()
        # Convolution 7x7 avec padding pour fusionner le pool moyen et max
        self.conv = nn.Conv2d(2, 1, 7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(x_cat)) # [cite: 334]

class CTSCAB(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.ca1 = ChannelAttention(in_channels)
        self.tsa = TimeSubcarrierAttention()
        self.ca2 = ChannelAttention(in_channels)

    def forward(self, x):
        x = x * self.ca1(x)
        x = x * self.tsa(x)
        x = x * self.ca2(x)
        return x # [cite: 311, 312, 313, 314]

class LI_HAR_Net(nn.Module):
    def __init__(self):
        super().__init__()
        layers = []
        in_c = 1
        
        # L'extracteur CNN est composé de 5 blocs successifs [cite: 280, 281]
        for _ in range(5):
            layers += [
                nn.Conv2d(in_c, 64, kernel_size=3, padding=1),
                nn.BatchNorm2d(64),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2)
            ]
            in_c = 64
            
        self.features = nn.Sequential(*layers)
        self.attention = CTSCAB(64) # Module d'attention [cite: 279, 280]

    def forward(self, x):
        x = self.features(x)
        x = self.attention(x)
        return x.view(x.size(0), -1) # Aplatissement en un vecteur d'embedding

# =====================================================================
# 4. Logique du Réseau Prototypique (Few-Shot Learning)
# =====================================================================
def euclidean_dist(x, y):
    """Calcule la distance euclidienne entre requêtes et prototypes"""
    n, m, d = x.size(0), y.size(0), x.size(1)
    x = x.unsqueeze(1).expand(n, m, d)
    y = y.unsqueeze(0).expand(n, m, d)
    return torch.pow(x - y, 2).sum(2)

def train_episode(model, optimizer, data, labels, n_way, n_support, n_query):
    """Gère un épisode complet d'entraînement (Support Set + Query Set)"""
    model.train()
    optimizer.zero_grad()
    
    classes = torch.unique(labels)
    # Sélection des classes pour cet épisode
    selected_cls = classes[torch.randperm(len(classes))[:n_way]]
    
    support_features, query_features = [], []
    query_labels = []

    for idx, c in enumerate(selected_cls):
        # Récupération de tous les échantillons de la classe 'c'
        cls_data = data[labels == c]
        
        # Sécurité au cas où une classe aurait très peu d'échantillons
        n_s = min(n_support, len(cls_data) // 2)
        n_q = min(n_query, len(cls_data) - n_s)
        
        if n_s == 0 or n_q == 0:
            continue
            
        perm = torch.randperm(len(cls_data))
        
        # Séparation du Support (pour le prototype) et Query (pour le test)
        sup = cls_data[perm[:n_s]]
        que = cls_data[perm[n_s:n_s+n_q]]
        
        # Le prototype est la moyenne des embeddings du support set
        support_features.append(model(sup).mean(0)) 
        query_features.append(model(que))
        query_labels.append(torch.full((n_q,), idx, dtype=torch.long))

    if not support_features:
        return 0.0, 0.0

    prototypes = torch.stack(support_features)
    query_features = torch.cat(query_features)
    query_labels = torch.cat(query_labels).to(device)
    
    # Calcul des distances et Log-Softmax
    dists = euclidean_dist(query_features, prototypes)
    log_p_y = F.log_softmax(-dists, dim=1)
    
    # Perte (Loss) et Rétropropagation
    loss = F.nll_loss(log_p_y, query_labels)
    loss.backward()
    optimizer.step()
    
    # Précision (Accuracy)
    acc = (log_p_y.argmax(1) == query_labels).float().mean()
    return loss.item(), acc.item()

# =====================================================================
# 5. Lancement de l'entraînement
# =====================================================================
if __name__ == '__main__':
    # Le chemin corrigé que nous avons vérifié ensemble
    dataset_path = 'NTU-Fi_HAR/train_amp'
    
    print(f"\n📂 Chargement des données depuis : {dataset_path}")
    train_dataset = NTUFiDataset(dataset_path)
    
    print(f"📊 Nombre total d'échantillons trouvés : {len(train_dataset)}")

    if len(train_dataset) == 0:
        print("\n❌ L'entraînement est annulé car le dataset est vide.")
    else:
        # DataLoader pour charger tout le dataset en mémoire
        train_loader = DataLoader(train_dataset, batch_size=len(train_dataset), shuffle=True)

        # Initialisation
        model = LI_HAR_Net().to(device)
        optimizer = optim.Adam(model.parameters(), lr=0.001) # [cite: 343, 344]

        print("\n🚀 Démarrage de l'entraînement (Few-Shot)...")
        epochs = 40 # [cite: 345]
        
        for epoch in range(epochs):
            for data, labels in train_loader:
                data, labels = data.to(device), labels.to(device)
                
                # 6 classes (n_way), 5-shot (n_support), 5 requêtes d'évaluation (n_query)
                loss, acc = train_episode(
                    model=model, 
                    optimizer=optimizer, 
                    data=data, 
                    labels=labels, 
                    n_way=6, 
                    n_support=5, 
                    n_query=5
                )
            
            # Affichage de la progression
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"Époque [{epoch+1}/{epochs}] | Loss: {loss:.4f} | Précision: {acc:.4f}")
                
        print("\n✅ Entraînement terminé avec succès !")

🖥️ Appareil utilisé : cpu

📂 Chargement des données depuis : NTU-Fi_HAR/train_amp
📊 Nombre total d'échantillons trouvés : 936

🚀 Démarrage de l'entraînement (Few-Shot)...
